# Розділ 7: Створення чат-додатків
## Швидкий старт з Microsoft Foundry Models API

Ця записна книжка адаптована з [репозиторію прикладів Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який включає записні книжки для доступу до сервісів [Azure OpenAI](notebook-azure-openai.ipynb).

> **Примітка:** GitHub Models буде припинено до кінця липня 2026 року. Ця записна книжка тепер орієнтована на [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), який пропонує той самий каталог моделей із безкоштовною можливістю тестування та досвід Azure AI Inference SDK.


# Огляд  
"Великі мовні моделі — це функції, які відображають текст у текст. Маючи вхідний рядок тексту, велика мовна модель намагається передбачити, який текст з’явиться наступним"(1). Ця записна книжка «швидкого старту» ознайомить користувачів із загальними поняттями великих мовних моделей, основними вимогами пакетів для початку роботи з AML, м’яким вступом у дизайн підказок і кількома короткими прикладами різних сценаріїв використання. 


## Зміст  

[Огляд](#overview)  
[Як використовувати сервіс OpenAI](#how-to-use-openai-service)  
[1. Створення вашого сервісу OpenAI](#1.-creating-your-openai-service)  
[2. Встановлення](#2.-installation)    
[3. Облікові дані](#3.-credentials)  

[Випадки використання](#use-cases)    
[1. Підсумовування тексту](#1.-summarize-text)  
[2. Класифікація тексту](#2.-classify-text)  
[3. Генерація нових назв продуктів](#3.-generate-new-product-names)  
[4. Тонке налаштування класифікатора](#4.fine-tune-a-classifier)  

[Посилання](#references)


### Створіть свій перший запит  
Ця коротка вправа забезпечить базове введення для надсилання запитів до моделі в Microsoft Foundry Models для простого завдання "резюмування".


**Кроки**:  
1. Встановіть бібліотеку `azure-ai-inference` у своєму python середовищі, якщо ви ще цього не зробили.  
2. Завантажте стандартні допоміжні бібліотеки та налаштуйте облікові дані Microsoft Foundry Models.  
3. Виберіть модель для свого завдання  
4. Створіть простий запит для моделі  
5. Надішліть свій запит до API моделі!


### 1. Встановіть `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Імпорт допоміжних бібліотек та створення облікових даних


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Вибір правильної моделі  
Моделі на кшталт GPT-4o та GPT-4o mini можуть розуміти та генерувати природну мову, і доступні у каталозі Microsoft Foundry Models разом із моделями від Meta, Mistral, Cohere та Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Проєктування запитів  

"Чарівність великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю помилку прогнозування на величезних обсягах тексту, моделі в кінцевому підсумку вивчають корисні концепції для цих прогнозів. Наприклад, вони вчаться таким концепціям"(1):

* як писати правильно
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як вести розмову
* як писати на багатьох мовах
* як кодувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх вхідних даних для великої мовної моделі найбільший вплив має текстовий запит(1).

Великі мовні моделі можна спонукати до створення виводу кількома способами:

Інструкція: Скажіть моделі, що ви хочете
Завершення: Спонукайте модель завершити початок того, що ви хочете
Демонстрація: Покажіть моделі, що ви хочете, маючи:
Кілька прикладів у запиті
Сотні або тисячі прикладів у навчальному наборі для донавчання"



#### Існує три основні рекомендації для створення запитів:

**Показуйте та пояснюйте**. Чітко вкажіть, що ви хочете, через інструкції, приклади або їх поєднання. Якщо ви хочете, щоб модель відсортувала список предметів в алфавітному порядку або класифікувала абзац за настроєм, покажіть їй, що саме ви хочете.

**Надавайте якісні дані**. Якщо ви намагаєтеся створити класифікатор або змусити модель слідувати певному шаблону, переконайтеся, що прикладів достатньо. Обов’язково перевіряйте свої приклади — модель зазвичай достатньо розумна, щоб побачити базові орфографічні помилки і дати відповідь, але вона також може припустити, що це зроблено навмисно, і це вплине на відповідь.

**Перевірте свої налаштування.** Параметри temperature і top_p контролюють, наскільки детермінованою є модель при формуванні відповіді. Якщо ви запитуєте відповідь, де є лише одна правильна, то вам слід встановити ці параметри нижче. Якщо ви шукаєте більш різноманітні відповіді, можливо, захочете їх підвищити. Найбільшою помилкою людей із цими налаштуваннями є те, що вони припускають, наче це контролі «кмітливості» або «креативності».


Джерело: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Відправити! 


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Повторіть той самий виклик, як порівнюються результати? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Резюмувати текст  
#### Виклик  
Резюмуйте текст, додавши 'tl;dr:' в кінці текстового уривка. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими підказками, ніж tl;dr, щоб змінити поведінку моделі та налаштувати отримане резюме(3).  

Останні дослідження продемонстрували значне покращення у багатьох NLP-завданнях та бенчмарках завдяки попередньому навчанню на великому корпусі тексту з подальшим донавчанням на конкретному завданні. Хоча зазвичай архітектура є завдань-незалежною, цей метод все одно потребує спеціалізованих датасетів для донавчання із тисячами або десятками тисяч прикладів. Натомість люди зазвичай можуть виконати нове мовне завдання, маючи лише кілька прикладів або прості інструкції — те, з чим сучасні NLP-системи досі значною мірою борються. Тут ми показуємо, що збільшення розміру мовних моделей значно покращує завдань-незалежну продуктивність у режимі few-shot, іноді навіть досягаючи конкуренції з попередніми передовими методами донавчання. 



Tl;dr


# Вправи для кількох випадків використання  
1. Підсумувати текст  
2. Класифікувати текст  
3. Створити нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Класифікація тексту  
#### Завдання  
Класифікуйте елементи у категорії, задані під час інференсу. У наступному прикладі ми надаємо як категорії, так і текст для класифікації в підказці(*playground_reference). 

Запит клієнта: Привіт, нещодавно зламалася одна з клавіш на клавіатурі мого ноутбука, і мені потрібна заміна:

Визначена категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Створення нових назв продуктів
#### Виклик
Створіть назви продуктів на основі прикладів слів. Тут ми включаємо у запит інформацію про продукт, для якого будемо генерувати назви. Ми також надаємо схожий приклад, щоб показати шаблон, який хочемо отримати. Ми також встановили високе значення температури, щоб збільшити випадковість і отримати більш інноваційні відповіді.

Опис продукту: Домашній пристрій для приготування молочних коктейлів
Початкові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, що підходить для будь-якого розміру стопи.
Початкові слова: адаптований, підходить, універсальний.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Посилання  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Найкращі практики тонкого налаштування моделей GPT для класифікації тексту](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[Команда з комерціалізації OpenAI](AzureOpenAITeam@microsoft.com) 


# Учасники
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
